# Chapter 2

In [1]:
# We need these as some functions have moved from Base to Stdlib
#
using Printf, SpecialFunctions, LinearAlgebra

┌ Info: Precompiling SpecialFunctions [276daf66-3868-5448-9aa4-cd146d93841b]
└ @ Base loading.jl:1278


### Arithmetic data types

In [2]:
x = 2;   typeof(x)

Int64

In [3]:
x = 2.0;  typeof(x)

Float64

In [4]:
x0 = 2^65
x1 = big(2)^65
@assert x0 == x1

LoadError: [91mAssertionError: x0 == x1[39m

In [5]:
for T = Any[Int8,Int16,Int32,Int64,Int128,UInt8,UInt16,UInt32,UInt64,UInt128]
    println("$(lpad(T,7)): [$(typemin(T)),$(typemax(T))]")
end

   Int8: [-128,127]
  Int16: [-32768,32767]
  Int32: [-2147483648,2147483647]
  Int64: [-9223372036854775808,9223372036854775807]
 Int128: [-170141183460469231731687303715884105728,170141183460469231731687303715884105727]
  UInt8: [0,255]
 UInt16: [0,65535]
 UInt32: [0,4294967295]
 UInt64: [0,18446744073709551615]
UInt128: [0,340282366920938463463374607431768211455]


### Logical and Bit datatypes

In [6]:
x = 0xbb31; y = 0xaa5f;  xor(x,y)

0x116e

In [7]:
x = 0xbb31;  x << 8

0x3100

In [8]:
x = 0xbb31; p = (2 < 3); x + p

0xbb32

### Arrays

In [9]:
# v1.0 requires the Statistics module for mean(), std() etc.
#
using Statistics

# Mean of 15 random numbers in range 0:100
#
A = rand(0:100,15)
mean(A)

41.6

In [10]:
# Create an empty array, note new syntax
# 'undef' will not initialise the elements
#
A = Array{Int64,1}(undef, 15)

# Verify: Tuple of the element type and the dimension sizes
#
(eltype(A),size(A))

(Int64, (15,))

In [11]:
# Fill array A with the first 15 Fibonnaci series
#
A[1] = 1
A[2] = 1
[A[i] = A[i-1] + A[i-2] for i = 3:length(A)]

13-element Array{Int64,1}:
   2
   3
   5
   8
  13
  21
  34
  55
  89
 144
 233
 377
 610

### Factorials

In [12]:
# The 'recursive' definition of factorial function
# A simple loop is much quicker
#
function fac(n::Integer)
  @assert n > 0
  (n == 1) ? 1 : n*fac(n-1)
end

fac (generic function with 1 method)

In [13]:
# This has difficulties with integer overflow
# We now need the Printf module to use the @printf macro
#
using Printf
for i = 1:30
  @printf "%3d : %d \n" i fac(i)
end

  1 : 1 
  2 : 2 
  3 : 6 
  4 : 24 
  5 : 120 
  6 : 720 
  7 : 5040 
  8 : 40320 
  9 : 362880 
 10 : 3628800 
 11 : 39916800 
 12 : 479001600 
 13 : 6227020800 
 14 : 87178291200 
 15 : 1307674368000 
 16 : 20922789888000 
 17 : 355687428096000 
 18 : 6402373705728000 
 19 : 121645100408832000 
 20 : 2432902008176640000 
 21 : -4249290049419214848 
 22 : -1250660718674968576 
 23 : 8128291617894825984 
 24 : -7835185981329244160 
 25 : 7034535277573963776 
 26 : -1569523520172457984 
 27 : -5483646897237262336 
 28 : -5968160532966932480 
 29 : -7055958792655077376 
 30 : -8764578968847253504 


In [14]:
# But since a BigInt <: Integer if we pass a BigInt the reoutine returns one
#
fac(big(30))


265252859812191058636308480000000

In [15]:
# Find stdlib, location is O/S dependent

cd(Sys.BINDIR)
cd("../share/julia/stdlib/v1.0")
pwd()

LoadError: [91mIOError: chdir ../share/julia/stdlib/v1.0: no such file or directory (ENOENT)[39m

In [16]:
# We can check this using the gamma function
# Again we need a module (SpecialFunctions)
#
using SpecialFunctions
gamma(31)     # Γ(n+1)  <=>  n!

2.6525285981219107e32

In [17]:
# This non-recursive one liner works!
# Note that this returns a BigInt regardless of the input
#
fac(N::Integer) = 
  (N < 1) ? throw(ArgumentError("N must be positive")) : reduce(*,big.(collect(1:N)))

@time(fac(402))

  0.008101 seconds (2.01 k allocations: 103.617 KiB)


10322493151921465164081017511444523549144957788957729070658850054871632028467255601190963314928373192348001901396930189622367360453148777593779130493841936873495349332423413459470518031076600468677681086479354644916620480632630350145970538235260826120203515476630017152557002993632050731959317164706296917171625287200618560036028326143938282329483693985566225033103398611546364400484246579470387915281737632989645795534475998050620039413447425490893877731061666015468384131920640823824733578473025588407103553854530737735183050931478983505845362197959913863770041359352031682005647007823330600995250982455385703739491695583970372977196372367980241040180516191489137558020294105537577853569647066137370488100581103217089054291400441697731894590238418118698720784367447615471616000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000

In [18]:
gamma(big(403.0))

1.032249315192146516408101751144452354914495778895772907065885005487163202846731e+874

---

### Other methods for generating Fibonacci numbers

In [19]:
# The 'standard' recursive definition

function fib(k::Integer)
  @assert k > 0
  (k < 3) ? 1 : fib(k-1) + fib(k-2)
end

@time fib(15)

  0.000008 seconds


610

In [20]:
# A better version

function fib(n::Integer)
  @assert n > 0
  a = Array{typeof(n),1}(undef,n)
  a[1] = 0
  a[2] = 1
  for i = 3:n
    a[i] = a[i-1] + a[i-2]
  end
  return a[n]
end

@time(fib(big(402)))


  0.000843 seconds (6.41 k allocations: 133.688 KiB)


284812298108489611757988937681460995615380088782304890986477195645969271404032323901

In [21]:
# A still better version
# This requires no array storage

function fib(n::Integer)
  @assert n > 0
  (a, b) = (big(0), big(1))
  while n > 0
    (a, b) = (b, a+b)
    n -= 1
  end
  return a
end


fib (generic function with 1 method)

In [22]:
# Golden ratio

@printf "%.15f" fib(101)/fib(100)


1.618033988749895

In [23]:
# Check with the actual value

γ = (1.0 + sqrt(5.0))/2.0

1.618033988749895

### Bulls and Cows

- This takes input from Standard Input (stdin)
- It does not run well within a Jupyter cell
- Better to cut-n-paste it into a terminal running a Julia REPL

In [ ]:
using Random # stdlib module needed for srand() => seed!()
#
tm = round(time());
seed = convert(Int64,tm);
Random.seed!(seed);

In [ ]:
function bacs()
         bulls = cows = turns = 0
         a = Any[]
         while length(unique(a)) < 4 
           push!(a,rand('0':'9'))
         end
         my_guess = unique(a)
         println("Bulls and Cows")
         while (bulls != 4)
           print("Guess? > ")
           s = chomp(readline(stdin))
           if (s == "q")
             print("My guess was "); [print(my_guess[i]) for i=1:4]
             return
           end
           guess = collect(s)
           if !(length(unique(guess)) == length(guess) == 4 && all(isdigit,guess))
             print("\nEnter four distinct digits or q to quit: ")
             continue
           end
           bulls = sum(map(==, guess, my_guess))
           cows = length(intersect(guess,my_guess)) - bulls
           println("$bulls bulls and $cows cows!")
           turns += 1
         end
         println("\nYou guessed my number in $turns turns.")
       end

In [ ]:
bacs()


---

### Cat and Mouse 

In [24]:
# The matrix file is in Files subdirectory
# Check we are at the correct location
#
cd() # Change as needed
pwd()

"/Users/malcolm"

In [25]:
#http://en.wikipedia.org/wiki/Stochastic_matrix
#
# Create stochastic matrix and write to disk
#
open("./cm3.txt","w") do f
  write(f,"0.0,0.0,0.5,0.0\n")
  write(f,"0.0,0.0,1.0,0.0\n")
  write(f,"0.25,0.25,0.0,0.25\n")
  write(f,"0.0,0.0,0.5,0.0\n")
end

16

In [26]:
using DelimitedFiles

I = zeros(4,4);
[I[i,i] = 1 for i in 1:4];

In [27]:
f = open("./cm3.txt","r")
T = readdlm(f,',');
close(f);

T

4×4 Array{Float64,2}:
 0.0   0.0   0.5  0.0
 0.0   0.0   1.0  0.0
 0.25  0.25  0.0  0.25
 0.0   0.0   0.5  0.0

In [28]:
Ep = [0 1 0 0]*inv(I - T)*[1,1,1,1];

println("Expected lifetime for the mouse is $(Ep[1]) hops.")

Expected lifetime for the mouse is 4.5 hops.


---

### Norms

_There is more than one way to skin a cat_ 

In [29]:
# Look at different definitions of the norm function
# For a Gaussian distribution of size N we should expect the answer ~= √N
# The first call f1(1) is to run in the function and not affect the timing
# This version uses the function in the stdlib LinearAlgebra module

using LinearAlgebra

f1(n) = norm(randn(n))
f1(10);
@time f1(100_000_000)


  1.280155 seconds (2 allocations: 762.940 MiB, 0.64% gc time)


9999.843423928385

In [30]:
# We can get the same result using a mapreduce procedure
# Note that it is a new set of random number, so the answer is slightly different
# The time is about the same

f2(n) = sqrt(mapreduce(x -> x*x, +, randn(n)))
f2(10);
@time f2(100_000_000)


  1.136198 seconds (2 allocations: 762.940 MiB, 24.86% gc time)


10000.140361435404

In [31]:
# Using a conventional mapping we need to pipe the result to sum it 
# and then take the square root
# This takes a little longer tha the previous 2

f3(n) = map(x -> x*x,randn(n)) |> sum |> sqrt
f3(10);
@time f3(100_000_000)


  2.247909 seconds (4 allocations: 1.490 GiB, 10.12% gc time)


9999.036199099404

In [32]:
# Finally we can non-vectorize the code, which is much quicker,
# In Julia non-vectorized (i.e loopy) code is invariably faster
# than the vectorized equivalent.

function f4(n)
    t = 0.0
    for i = 1:n
        t += randn()^2
    end
    return sqrt(t)
end

f4(10);
@time f4(100_000_000)


  0.981226 seconds


10000.604074285056

### Pointy Norms

In [33]:
# Define a very simple type to represent a 2-D point

struct Point
  x::Real
  y::Real
end


In [34]:
# We can define how to add and scale points
# This needs importing the + and * function from Base

import Base: +,*
+(u::Point,v::Point) = Point(u.x + v.x, u.y + v.y )
*(a::Real,u::Point)  = Point(a*u.x, a*u.y)
*(u::Point,a::Real)  = a*u


* (generic function with 416 methods)

In [35]:
# Just test the type structure

u1 = Point(1.0,2.3)
u1*(17//13)


Point(1.3076923076923077, 3.0076923076923077)

In [36]:
# Using Julia's aggregate object model this type "knows" all about arrays
# Note: I'll deal with the object model in greater detail the next two chapters
#
aa = [Point(randn(),randn()) for i = 1:100];
ab = reshape(aa,10,10)


10×10 Array{Point,2}:
 Point(-1.87005, 0.277947)    …  Point(0.213591, -0.526014)
 Point(0.259048, 0.40708)        Point(-0.337676, 0.997118)
 Point(1.05337, -0.274245)       Point(0.242773, -1.22169)
 Point(-1.08857, -1.42577)       Point(0.0694403, -0.366422)
 Point(0.491453, 0.845923)       Point(-0.143337, 0.553089)
 Point(2.18334, 0.391378)     …  Point(-0.171449, -0.306596)
 Point(-0.836745, 1.14644)       Point(-0.699404, 0.383505)
 Point(0.739549, -0.392151)      Point(0.611134, -1.97815)
 Point(-0.373328, -0.558464)     Point(-1.48648, -0.595363)
 Point(0.936538, -0.925918)      Point(-0.16457, -1.33168)

In [37]:
# It is useful to define a zero function (not sure about a one())

zero(Point) = Point(0.0,0.0)


zero (generic function with 1 method)

In [38]:
# The dot product is the sum of the product of the x and y coordinates

dot(u::Point)::Real = u.x^2 * u.y^2
dot(u::Point,v::Point)::Real = u.x*v.x * u.y*v.y
dot(ab[1])

0.2701655814143984

In [39]:
# The distance between two points is determined by Pythagoras's rule

dist(u::Point,v::Point)::Real = sqrt((u.x - v.x)^2 + (u.y - v.y)^2)
dist(ab[4,1],ab[2,7])

4.132923799983675

In [40]:
# The distance of the point from the origin is equivalent to it's norm 
# We need to import the 'norm' function

import LinearAlgebra.norm

norm(u::Point)::Real = sqrt(u.x^2 + u.y^2)
norm(aa[17])


0.9298625595804534

In [41]:
# It is also possible to define this as:

dist(u::Point)::Real = dist(u::Point,zero(Point))

# Although this requires slightly more work to compute and it may be
# better just to define it as dist(u::Point) = norm(u)

@assert(dist(aa[17]) == norm(aa[17]))  ## Should produce NO output if it is TRUE


In [42]:
typeof(aa)  # Note that this is an array of points
            # So norm will not work on this yet


Array{Point,1}

In [43]:
# One way is to overload the norm function using the mapreduce function above
# (or write a non-vectorized one)

norm(a::Array{Point,1}) = sqrt(mapreduce(x -> dist(x), +, a))


norm (generic function with 14 methods)

In [44]:
# And now we can estimate PI (as in the previous chapter)

N = 1000000; ac = [Point(rand(),rand()) for i = 1:N];

# We need the let/end because of the scoping rules
let
  count = 0
  for i = 1:N
    (dist(ac[i]) < 1.0) && (count += 1)
  end
  4.0*(count/N)
end

3.140644

In [45]:
# Note as yet norm(ab) will not work so we need to be a little more imaginative
# with the function definition. 

# In the next chapter we will generalise this type definition to 3-Vector and N-vector
# and revisit the quiestion then

norm(ab)


14.061963084722013

---

### Generate a *Julia* set

In [46]:
function juliaset(z, z0, nmax::Int64)
    for n = 1:nmax
        if abs(z) > 2 (return n-1) end
        z = z^2 + z0
    end
    return nmax
end


juliaset (generic function with 1 method)

In [47]:
function create_pgmfile(img, outf::String)
    s = open(outf, "w")
    write(s, "P5\n")    
    n, m = size(img)
    write(s, "$m $n 255\n")
    for i=1:n, j=1:m
        p = min(img[i,j],255)
        write(s, UInt8(p))
    end
    close(s)
end


create_pgmfile (generic function with 1 method)

In [48]:
h = 400; 
w = 800; 
m = Array{Int64,2}(undef,h,w);

c0 = -0.8 + 0.16im;
pgm_name = "jset.pgm";

t0 = time();
for y=1:h, x=1:w
    c = complex((x-w/2)/(w/2), (y-h/2)/(w/2))
    m[y,x] = juliaset(c, c0, 256)
end
t1 = time();


In [49]:
# You should find the file in the same chapter as the notebook

create_pgmfile(m, pgm_name);
@printf "Written %s\nFinished in %.4f seconds.\n" pgm_name (t1-t0);


Written jset.pgm
Finished in 0.6206 seconds.


In [50]:
# Display the image using Imagemagick's display command
# Clicking the file may display it (OSX/XQuartz and Centos/Gnome certainly does)
# On OSX use:  brew reinstall imagemagick --with-x11
#
# Otherwise you make be able to click onit (OSX or Linux)
# or on WIndows use an image processing program such as Irfanview
# We will be looking at how to do this in Julia later.

here = pwd()
run(`display $here/$pgm_name`)

Version: ImageMagick 7.0.10-37 Q16 x86_64 2020-11-09 https://imagemagick.org
Copyright: © 1999-2020 ImageMagick Studio LLC
License: https://imagemagick.org/script/license.php
Features: Cipher DPC HDRI Modules OpenMP(4.5) 
Delegates (built-in): bzlib freetype gslib heic jng jp2 jpeg lcms lqr ltdl lzma openexr png ps tiff webp xml zlib
Usage: display [options ...] file [ [options ...] file ...]

Image Settings:
  -alpha option        on, activate, off, deactivate, set, opaque, copy
                       transparent, extract, background, or shape
  -antialias           remove pixel-aliasing
  -authenticate password
                       decipher image with this password
  -backdrop            display image centered on a backdrop
  -channel type        apply option to select image channels
  -colormap type       Shared or Private
  -colorspace type     alternate image colorspace
  -comment string      annotate image with comment
  -compress type       type of pixel compression when writi

display: delegate library support not built-in '' (X11) @ error/display.c/DisplayImageCommand/1889.


LoadError: [91mfailed process: Process(`[4mdisplay[24m [4m/Users/malcolm/jset.pgm[24m`, ProcessExited(1)) [1][39m


---